In [ ]:
"""
    Code to plot all UKB and AoU main figures
"""

import pandas as pd
import numpy as np

import os 
from options.options import Options
import util.util as util
import util.visual as vis
import util.bootstrap_tools as btool

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as plticker

from scipy.stats import norm, pearsonr, spearmanr

results_dir = 'results'

opt = Options()
opt.initialize()

bin_defs = opt.bin_defs
phen_labels = ['AFIB', 'CAD', 'T2D', 'HCA', 'AST', 'OB', 'MDD']
phen_labels_title = ['AFIB', 'CAD', 'T2D', 'HC', 'AST', 'OB', 'MDD']


covars = pd.read_csv('covars_df.csv', sep=',', index_col='IID')[opt.covars]
bounds, labels, label_map = util.create_bins_and_indices(covars, bin_defs)

# Load files for each cohort

In [ ]:
cohort_keys = ['eur', 'afr', 'ukb']

all_dict = {k: {} for k in cohort_keys}
or_dict = {k: {} for k in cohort_keys}
top_or_dict = {k: {} for k in cohort_keys}
top_ar_dict = {k: {} for k in cohort_keys}
top_ar_null_dict = {k: {} for k in cohort_keys}
r2_liab_dict = {k: {} for k in cohort_keys}
prev_dict = {k: {} for k in cohort_keys}
bootstrap_dict = {k: {} for k in cohort_keys}
minmax_3way_dict = {k: {} for k in cohort_keys}
results_3way_dict = {k: {} for k in cohort_keys}
pdiff_dict = {k: {} for k in cohort_keys}
income_dict = {k: {} for k in cohort_keys}
high_pgs_dict = {k: {} for k in cohort_keys}
n_dict = {k: {} for k in cohort_keys}


top_dict = {k: {} for k in cohort_keys}
bot_dict = {k: {} for k in cohort_keys}

top_diff_dict = {k: {} for k in cohort_keys}
bot_diff_dict = {k: {} for k in cohort_keys}

pzero_dict = {k: {} for k in cohort_keys}
pdiff_dict = {k: {} for k in cohort_keys}

pval_zero = {k: {} for k in cohort_keys}
pval_parent = {k: {} for k in cohort_keys}

for anc in ['eur', 'afr']:
    for phen in phen_labels:
        all_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/all_results.npy', allow_pickle=True).item()
        or_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/or.npy')
        top_or_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/top_or.npy')

        top_ar_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/top_ar.npy')
        top_ar_null_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/top_ar_null.npy')
        prev_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/prev.npy')
        
        r2_liab_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/r2_liab.npy')
        
        n_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/n_idx.npy')
        high_pgs_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/high_pgs.npy')

        bootstrap_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/bootstrap.npy', allow_pickle=True).item()

        pdiff_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/odds_ratio_pdiff_2way.npy')
        
        top_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/top_results.npy', allow_pickle=True).item()
        bot_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/bot_results.npy', allow_pickle=True).item()
        
        top_diff_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/top_ar_diff.npy', allow_pickle=True).item()
        bot_diff_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/bot_ar_diff.npy', allow_pickle=True).item()
        
        pval_zero[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/pval_zero.npy')
        pval_parent[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/pval_parent.npy')
        
        income_dict[anc][phen] = np.load(f'{results_dir}/{phen}/{anc}/low_income.npy', allow_pickle=True).item()
        
all_dict['ukb'] = np.load(f'{results_dir}/all_dict_UKB.npy', allow_pickle=True).item()
top_or_dict['ukb'] = np.load(f'{results_dir}/top_or_dict_UKB.npy', allow_pickle=True).item()
top_ar_dict['ukb'] = np.load(f'{results_dir}/top_ar_dict_UKB.npy', allow_pickle=True).item()
top_ar_null_dict['ukb'] = np.load(f'{results_dir}/top_ar_null_dict_UKB.npy', allow_pickle=True).item()
prev_dict['ukb'] = np.load(f'{results_dir}/prev_dict_UKB.npy', allow_pickle=True).item()
pdiff_dict['ukb'] = np.load(f'{results_dir}/pdiff_dict_UKB.npy', allow_pickle=True).item()

top_dict['ukb'] = np.load(f'{results_dir}/top_results.npy', allow_pickle=True).item()
bot_dict['ukb'] = np.load(f'{results_dir}/bot_results.npy', allow_pickle=True).item()

top_diff_dict['ukb'] = np.load(f'{results_dir}/top_diff_results.npy', allow_pickle=True).item()
bot_diff_dict['ukb'] = np.load(f'{results_dir}/bot_diff_results.npy', allow_pickle=True).item()

pval_zero['ukb'] = np.load(f'{results_dir}/pval_zero_UKB.npy', allow_pickle=True).item()
pval_parent['ukb'] = np.load(f'{results_dir}/pval_parent_UKB.npy', allow_pickle=True).item()


n = len(opt.covars)
n_context = n*(n+1)/2

n_strata_diag = [2,3,3,2,3,3]
n_strata = np.outer(n_strata_diag, n_strata_diag)
np.fill_diagonal(n_strata, n_strata_diag)

pval_zero_summary = {'eur': np.full((len(phen_labels), n, n), 0), 
                     'afr': np.full((len(phen_labels), n, n), 0),
                     'ukb': np.full((len(phen_labels), n, n), 0)}
pval_parent_summary = {'eur': np.full((len(phen_labels), n, n), 0), 
                     'afr': np.full((len(phen_labels), n, n), 0),
                     'ukb': np.full((len(phen_labels), n, n), 0)}
for anc in ['eur', 'afr', 'ukb']:
    for i, phen in enumerate(phen_labels):
        pval_zero_ = pval_zero[anc][phen]<.05/(n_context*n_strata)
        pval_zero_summary[anc][i] = pval_zero_
        pval_parent_summary[anc][i] = pval_parent[anc][phen]<.05/2 

# Save results into an excel sheet

In [ ]:
%pip install XlsxWriter

## Supplemental Table 5

In [ ]:
columns = ['Context 1', 'Context 2', 
           'OR', 'AR (%)', 'AR Prev. Adj. (%)', 
           'Disease Prevalence (%)', '% of High-PGS Individuals', '# of Samples',
           'OR', 'AR (%)', 'AR Prev. Adj. (%)', 
           'Disease Prevalence (%)', '% of High-PGS Individuals', '# of Samples']

with pd.ExcelWriter("risk_across_contexts.xlsx", engine="xlsxwriter") as writer:
    for phen in phen_labels:
        rows = []
        
        res = []
        for anc in ['eur', 'afr']:
            res += [all_dict[anc][phen]['top_or'], 100*all_dict[anc][phen]['top_ar'], 
                     None, 100*all_dict[anc][phen]['prev'], 100*all_dict[anc][phen]['high_pgs'], 
                    all_dict[anc][phen]['n']]
            
        rows.append([None, None]+ res)
        
        for i, label in enumerate(opt.labels_shorthand):
            label1 = opt.labels_shorthand[i]
            label2 = None
            
            res = []
            for anc in ['eur', 'afr']:
                top_or = top_or_dict[anc][phen][i,i]
                top_ar = 100*top_ar_dict[anc][phen][i,i]
                top_ar_null = 100*top_ar_null_dict[anc][phen][i,i]
                prev = 100*prev_dict[anc][phen][i,i]
                high_pgs = 100*high_pgs_dict[anc][phen][i,i]
                n_samps = int(n_dict[anc][phen][i,i])
                res += [top_or, top_ar, top_ar_null, prev, high_pgs, n_samps]
                
            rows.append([label1, label2]+res)

        for i, label in enumerate(opt.labels_shorthand):
            for j in range(i+1, len(opt.labels_shorthand)):
                label1 = opt.labels_shorthand[i]
                label2 = opt.labels_shorthand[j]
                
                res = []
                for anc in ['eur', 'afr']:
                    top_or = top_or_dict[anc][phen][i,j]
                    top_ar = 100*top_ar_dict[anc][phen][i,j]
                    top_ar_null = 100*top_ar_null_dict[anc][phen][i,j]
                    prev = 100*prev_dict[anc][phen][i,j]
                    high_pgs = 100*high_pgs_dict[anc][phen][i,j]
                    n_samps = int(n_dict[anc][phen][i,j])
                    res += [top_or, top_ar, top_ar_null, prev, high_pgs, n_samps]
                if int(n_dict['eur'][phen][i,j]) == 0:
                        continue
                rows.append([label1, label2]+res )

        df = pd.DataFrame(rows, columns=columns)
        df.iloc[:, 3:] = df.iloc[:, 3:].round(3)
        df.iloc[:, 3:] = df.iloc[:, 3:].round(3)
        df.to_excel(writer, sheet_name=phen[:31], index=False)

## Supplemental Table S6

In [ ]:
columns = ['Variable 1', 'Variable 2', 
           'Max. OR %Δ', '95% CI: Low', '95% CI: High', 'P-value', 'P-value (Intx.)',
           'Max. OR %Δ', '95% CI: Low', '95% CI: High', 'P-value', 'P-value (Intx.)']

with pd.ExcelWriter("percent_difference_across_variables.xlsx", engine="xlsxwriter") as writer:
    for phen in phen_labels:

        rows = []
        for i, var in enumerate(covars):
            label1 = opt.covars_sync[i]
            label2 = None
            
            res = []
            for anc in ['eur', 'afr']:
            
                pdiff = pdiff_dict[anc][phen][i,i][0]
                pdiff_ci = util.compute_ci(pdiff_dict[anc][phen][i,i][1:])

                pval_zero_ = pval_zero[anc][phen][i,i]
                pval_parent_ = pval_parent[anc][phen][i,i]
                res += [pdiff, pdiff_ci[0], pdiff_ci[1], pval_zero_, pval_parent_]
            rows.append([label1, label2]+res)

        for i, var in enumerate(opt.covars):
            for j in range(i+1, len(opt.covars)):
                label1 = opt.covars_sync[i]
                label2 = opt.covars_sync[j]
                
                res = []
                for anc in ['eur', 'afr']:
                
                    pdiff = pdiff_dict[anc][phen][i,j][0]
                    pdiff_ci = util.compute_ci(pdiff_dict[anc][phen][i,j][1:])

                    pval_zero_ = pval_zero[anc][phen][i,j]
                    pval_parent_ = pval_parent[anc][phen][i,j]
                    res += [pdiff, pdiff_ci[0], pdiff_ci[1], pval_zero_, pval_parent_]
                rows.append([label1, label2]+res)

        df = pd.DataFrame(rows, columns=columns)
        df.iloc[:, 2:] = df.iloc[:, 2:].round(3)
        df.to_excel(writer, sheet_name=phen[:31], index=False)

# UKB Eur. Vs. AoU Eur. ORs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define colors and markers for each phenotype
phen_colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']
phen_markers = ['o', 's', '^', 'D', 'P', 'X', '*']

plt.figure(figsize=(10, 6))

for i, phen in enumerate(phen_labels):
    or_ukb = util.extract_2way_interactions(top_or_dict['ukb'][phen])
    or_eur = util.extract_2way_interactions(top_or_dict['eur'][phen])

    plt.scatter(or_eur, or_ukb, 
                color=phen_colors[i], 
                marker=phen_markers[i], 
                label=opt.phen_params[phen]['title'])

plt.xlabel("OR (AoU; Eur.)", fontsize=18)
plt.ylabel("OR (UKB; Eur.)", fontsize=18)
plt.title("ORs Across Two-Way Contexts:", fontsize=24, fontweight='bold',y=1.08)
plt.suptitle("AoU Eur. vs. UKB Eur.", fontsize=22,y=.94)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.legend(title="Phenotypes", title_fontsize=18, fontsize=16, loc="best")
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig(f'{results_dir}/aou_ukb_or.png', bbox_inches='tight')
plt.show()

# OR correlations across cohorts

In [ ]:
from scipy.stats import norm, pearsonr, spearmanr
import numpy as np

def compute_fisher_ci(r, n, alpha=0.05):
    if np.abs(r) == 1:  # Avoid division by zero for perfect correlations
        return r, r
    z = np.arctanh(r)  # Fisher transformation
    se = 1 / np.sqrt(n - 3)  # Standard error
    z_critical = norm.ppf(1 - alpha / 2)  # Correct z-critical value
    ci = np.tanh([z - z_critical * se, z + z_critical * se])  # Transform back
    return ci[0], ci[1]

def process_correlations(two_way_afr, two_way_eur, two_way_ukb):
    results = {'correlations': {}, 'confidence_intervals': {}, 'pval': {}}
    pairs = {
        'afr_eur': (two_way_afr, two_way_eur),
        'afr_ukb': (two_way_afr, two_way_ukb),
        'eur_ukb': (two_way_eur, two_way_ukb)
    }
    for key, (x, y) in pairs.items():
        valid = ~np.isnan(x) & ~np.isnan(y)
        if np.any(valid):
            r, p = spearmanr(x[valid], y[valid])
            if not np.isnan(r):
                ci = compute_fisher_ci(r, np.sum(valid))
            else:
                ci = (np.nan, np.nan)  # Handle undefined correlation case
            results['correlations'][key] = r
            results['confidence_intervals'][key] = ci
            results['pval'][key] = p
    return results

import matplotlib.pyplot as plt

def plot_correlations_with_ci(correlations, confidence_intervals, pval, phen_labels, title=''):
    groups = list(correlations.keys())
    group_names = {'afr_eur': 'AoU (Eur.) vs. AoU (Afr.)',
                   'afr_ukb': 'UKB (Eur.) vs. AoU (Afr.)',
                   'eur_ukb': 'UKB (Eur.) vs. AoU (Eur.)'}
    
    x = np.arange(len(phen_labels))  # Phenotype indices
    width = 0.25  # Bar width
    offset = [-width, 0, width]  # Bar offsets for grouped plots

    # Color palette for bars
    colors = plt.cm.tab10(np.linspace(0, 1, len(groups)))

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.grid(axis='y',zorder=0)
    for i, group in enumerate(groups):
        means = correlations[group]
        cis = np.array(confidence_intervals[group])
        lower_errors = means - cis[:, 0]
        upper_errors = cis[:, 1] - means
        bars = ax.bar(
            x + offset[i],
            means,
            width=width,
            label=group_names[group],
            yerr=[lower_errors, upper_errors],
            capsize=5,
            color=colors[i],
            edgecolor="black",
            alpha=0.9,
            zorder=2
        )
        
        # Add significance stars above bars
        for j, bar in enumerate(bars):
            bar_x = bar.get_x() + bar.get_width() / 2
            bar_height = bar.get_height()
            p = pval[group][j]
            
            if p < 0.001/(3):
                star = "***"
            elif p < 0.01/(3):
                star = "**"
            elif p < 0.05/(3):
                star = "*"
            else:
                star = ""
            
            if star:
                ax.text(
                    bar_x,
                    cis[j, 1] + 0.01,  # Slightly above the upper bound of CI
                    star,
                    ha="center",
                    va="bottom",
                    fontsize=12,
                    weight="bold",
                    color="black",
                )

    # Aesthetics and labels
    #ax.set_xlabel("Phenotypes", fontsize=12, weight="bold")
    ax.set_ylabel("Spearman’s Corr.", fontsize=20)
    ax.set_title(title, fontsize=26, weight="bold", pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(phen_labels, rotation=90, ha="center", fontsize=18)
    ax.axhline(0, color='gray', linestyle='-')#, linewidth=0.7, alpha=0.7)
    ax.legend(fontsize=18,loc='lower left', frameon=True)
    ax.set_ylim([-1, 1])
    ax.tick_params(axis='y', labelsize=18) 
    
    ax.set_yticks(np.arange(-1, 1.01, 0.25))
    ax.grid(axis='y', which='major', linestyle='-', linewidth=0.8, alpha=0.7, zorder=0)
    

    # Adjust layout for readability
    plt.tight_layout()
    plt.savefig(f'{results_dir}/or_corrs_across_cohorts.png')
    plt.show()

## calculate OR correlation

In [ ]:
n_bins = len(labels)
ltril_inds = np.tril_indices(n_bins, -1)
two_way_all = {'afr': [], 'eur': [], 'ukb': []}
correlations = {'eur_ukb': [], 'afr_ukb': [], 'afr_eur': [],}
confidence_intervals = {'afr_eur': [], 'afr_ukb': [], 'eur_ukb': []}
pvals = {'afr_eur': [], 'afr_ukb': [], 'eur_ukb': []}
for phen in phen_labels:
    two_way = {
        'afr': top_or_dict['afr'][phen][ltril_inds]-np.nanmean(top_or_dict['afr'][phen][ltril_inds]),
        'eur': top_or_dict['eur'][phen][ltril_inds]-np.nanmean(top_or_dict['eur'][phen][ltril_inds]),
        'ukb': top_or_dict['ukb'][phen][ltril_inds]-np.nanmean(top_or_dict['ukb'][phen][ltril_inds])
    }
    
    for key in two_way_all:
        two_way_all[key].extend(two_way[key])
    phen_results = process_correlations(two_way['afr'], two_way['eur'], two_way['ukb'])
    for key in phen_results['correlations']:
        correlations[key].append(phen_results['correlations'][key])
        confidence_intervals[key].append(phen_results['confidence_intervals'][key])
        pvals[key].append(phen_results['pval'][key])

two_way_all = {key: np.array(values) for key, values in two_way_all.items()}
overall_results = process_correlations(two_way_all['afr'], two_way_all['eur'], two_way_all['ukb'])
for key in overall_results['correlations']:
    correlations[key].append(overall_results['correlations'][key])
    confidence_intervals[key].append(overall_results['confidence_intervals'][key])
    pvals[key].append(overall_results['pval'][key])

In [ ]:
plot_correlations_with_ci(correlations, confidence_intervals, pvals, phen_labels_title+['Overall\n(Mean Adj.)'], 'Correlation of ORs Across Cohorts')

# Variable Significance

In [ ]:
n_phen = len(phen_labels)

one_way_sig_var_dict = {k: np.zeros((n_phen, 6)) for k in cohort_keys}
two_way_sig_var_dict = {k: np.zeros((n_phen, 6)) for k in cohort_keys}
two_way_robust_sig_var_dict = {k: np.zeros((n_phen, 6)) for k in cohort_keys}

one_way_sig_phen_dict = {k: np.zeros(n_phen) for k in cohort_keys}
two_way_sig_phen_dict ={k: np.zeros(n_phen) for k in cohort_keys}
two_way_robust_sig_phen_dict = {k: np.zeros(n_phen) for k in cohort_keys}

for anc in cohort_keys:
    for p in range(n_phen):
        one_way_sig_phen_dict[anc][p] = np.trace(pval_zero_summary[anc][p])
        two_way_sig_phen_dict[anc][p] = ((pval_zero_summary[anc][p].sum(0) - np.diag(pval_zero_summary[anc][p]))>0).sum()
        two_way_robust_sig_phen_dict[anc][p] = ((pval_parent_summary[anc][p].sum(0) - np.diag(pval_parent_summary[anc][p]))>0).sum()

        for i in range(6):
            sig = pval_zero_summary[anc][p, i]
            robust_sig = pval_parent_summary[anc][p, i]

            one_way_sig_var_dict[anc][p, i] = sig[i]
            two_way_sig_var_dict[anc][p, i] = (sig.sum() - sig[i]) > 0
            two_way_robust_sig_var_dict[anc][p, i] = (robust_sig.sum() - robust_sig[i]) > 0

In [ ]:
sub_titles = {'ukb': '(UKB; Eur. Anc.)',
              'eur': '(AoU; Eur. Anc.)',
              'afr': '(AoU; Afr. Anc.)'}

# Combine three plots into a single figure with horizontally stacked subplots
fig, axes = plt.subplots(1, 3, figsize=(10, 10), sharey=False, gridspec_kw={'width_ratios': [.5, .5, .5]})

# Data for the first plot
xlabels_reversed_1 = ['Sex', 'Age', 'Alcohol Intake', 'Smoking History', 'House. Income', 'Towns. Depriv.'][::-1]

for ax_i, anc in enumerate(['ukb', 'eur', 'afr']):
    one_way_sig_var = one_way_sig_var_dict[anc]
    two_way_sig_var = two_way_sig_var_dict[anc]
    two_way_robust_sig_var = two_way_robust_sig_var_dict[anc]

    one_way_sig_phen = one_way_sig_phen_dict[anc]
    two_way_sig_phen = two_way_sig_phen_dict[anc]
    two_way_robust_sig_phen = two_way_robust_sig_phen_dict[anc]
  
    # Adjust the grid height for thinner blank rows
    rows, cols = one_way_sig_var.shape
    row_height = 1  # Adjust this for thinner blank rows
    spacing = 0.4  # Adjust this for spacing between sections
    grid_height = rows * (2 * row_height + spacing)  # 2 rows per variable + blank space
    grid_width = cols

    # Third plot: Blank grid visualization
    axes[ax_i].set_xlim(0, grid_width)
    axes[ax_i].set_ylim(-.5, grid_height + .75)
    axes[ax_i].tick_params(left=False, bottom=False, labelleft=True, labelbottom=False)

    # Add vertical gridlines for x-axis only
    axes[ax_i].set_xticks(range(grid_width))
    axes[ax_i].set_yticks([])
    axes[ax_i].grid(visible=True, which='major', axis='x', linestyle='-', linewidth=1, zorder=4)
    axes[ax_i].set_title(sub_titles[anc], fontsize=24, y= 1.01)

    # Add y-axis labels
    if ax_i == 0:
        y_labels = phen_labels  # Replace with actual labels
        for i, label in enumerate(y_labels):
            y_position = grid_height - i * (2 * row_height + spacing) - row_height / 2
            axes[ax_i].text(-0.1, y_position-.6, opt.phen_params[label]['title'], ha='right', va='center', fontsize=24)

    # Draw the grid with coloring
    for i in range(rows):
        y_base = grid_height - i * (2 * row_height + spacing)

        # First row color: coral for one_way_sig_var > 0
        for j in range(cols):
            if one_way_sig_var[i, j] > 0:
                axes[ax_i].add_patch(plt.Rectangle((j, y_base - row_height), 1, row_height, color='salmon', edgecolor='black'))

        # Second row color: lightblue for two_way_sig_var > 0
        for j in range(cols):
            if two_way_robust_sig_var[i, j] > 0:
                axes[ax_i].add_patch(plt.Rectangle((j, y_base - 2 * row_height), 1, row_height, color='#00509e', edgecolor='black'))
            elif two_way_sig_var[i, j] > 0:
                axes[ax_i].add_patch(plt.Rectangle((j, y_base - 2 * row_height), 1, row_height, color='#99ccff', edgecolor='black'))

# Add x-axis labels
xlabels = ['Sex', 'Age', 'Alc. Int.', 'Smok. His.', 'Inc.', 'Depriv.']
y_len = [-.55, -.6, -1.4, -2.1, -.3, -1.1]
for i in range(3):
    for j in range(cols):
        axes[i].text(j + 0.7, y_len[j] - 1, xlabels[j], ha='right', va='center', fontsize=24, rotation=70)

# Shared legend
from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor='salmon', edgecolor='none', label='One-Way (**)'),
    Patch(facecolor='#99ccff', edgecolor='none', label='Two-Way (**)'),
    Patch(facecolor='#00509e', edgecolor='none', label='Two-Way Intx. (⨂)')
]

fig.legend(handles=legend_handles, loc='lower center', fontsize=20, ncol=3, bbox_to_anchor=(0.5, .06))
fig.suptitle(f'Variable Significance', fontsize=30, y=.98, fontweight='bold')

# Adjust layout and save
plt.tight_layout(rect=[0, 0.1, 1, 1])
plt.savefig(f'{results_dir}/var_significance.png', bbox_inches='tight')
plt.show()

## Significant variable overlap between UKB Eur. and Aou Eur.

In [ ]:
from matplotlib_venn import venn2
import matplotlib.pyplot as plt
import numpy as np

n_bins = 6
ltril_inds = np.tril_indices(n_bins, -1)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

eur_n = 0
shared_n = 0
ukb_n = 0
for i in range(8):  # Iterate over 7 phenotypes
    phen_name = (phen_labels_title+['Overall'])[i]
    if i<7:
        eur = pval_zero_summary['eur'][i][ltril_inds]
        ukb = pval_zero_summary['ukb'][i][ltril_inds]
    if i == 7:
        eur = []
        ukb = []
        for j in range(7):
            eur += list(pval_zero_summary['eur'][j][ltril_inds])
            ukb += list(pval_zero_summary['ukb'][j][ltril_inds])
        eur = np.asarray(eur)
        ukb = np.asarray(ukb)

    # Calculate sets for Venn diagram
    eur_set = set(np.where(eur == 1)[0])
    ukb_set = set(np.where(ukb == 1)[0])
    shared_set = eur_set & ukb_set

    subsets = (len(eur_set - ukb_set), len(ukb_set - eur_set), len(shared_set))
    
    # Create Venn diagram
    venn = venn2(subsets=subsets, set_labels=('AoU (Eur.)', 'UKB (Eur.)'), ax=axes[i])
    eur_n += subsets[0]
    ukb_n += subsets[1]
    shared_n += subsets[2]

    # Adjust text and title visibility
    for idx, text in enumerate(venn.subset_labels):
        if text and subsets[idx] == 0:
            text.set_visible(False)

    for text in venn.set_labels:
        if text.get_text() in ['AoU (Eur.)', 'UKB (Eur.)']:
            if (subsets[0] == 0) and (subsets[2] == 0):
                text.set_visible(False)
                   
    for text in venn.set_labels:
        if text:
            text.set_fontsize(24) 
                
    for text in venn.subset_labels:
        if text:
            text.set_fontsize(30)
    
    if subsets[0] == 0 and subsets[1] == 0 and subsets[2] == 0:
        axes[i].set_title(f"{phen_name}", fontsize=40, y=.95)
    else:
        axes[i].set_title(f"{phen_name}", fontsize=40, y=.95)

# Remove unused subplot
if len(axes) > 8:
    for j in range(8, len(axes)):
        fig.delaxes(axes[j])

fig.suptitle(f'Shared Significant Variable Intersections (**)', fontsize=46, y=.975, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{results_dir}/venn.png', bbox_inches='tight')

plt.show()

# Concordance and Discordance

In [ ]:
plt.rcParams.update({'font.size': 20})
def pad_and_concat(odds_matrix, odds_side, odds_top):
    padded_matrix = np.pad(odds_matrix, ((2, 0), (2, 0)), constant_values=np.nan)
    padded_matrix[2:, 0] = odds_side
    padded_matrix[0, 2:] = odds_top
    return padded_matrix

phen = 'AST'
i = 2
j = 1

i_strt = bounds[i]
i_end = bounds[i+1]

j_strt = bounds[j]
j_end = bounds[j+1]

odds_ukb_ = top_or_dict['ukb'][phen][i_strt:i_end, j_strt:j_end]
odds_aou_eur_ = top_or_dict['eur'][phen][i_strt:i_end, j_strt:j_end]
odds_aou_afr_ = top_or_dict['afr'][phen][i_strt:i_end, j_strt:j_end]

odds_ukb_ = pad_and_concat(odds_ukb_,  np.diag(top_or_dict['ukb'][phen][i_strt:i_end, i_strt:i_end]), np.diag(top_or_dict['ukb'][phen][j_strt:j_end, j_strt:j_end]))
odds_aou_eur_ = pad_and_concat(odds_aou_eur_, np.diag(top_or_dict['eur'][phen][i_strt:i_end, i_strt:i_end]), np.diag(top_or_dict['eur'][phen][j_strt:j_end, j_strt:j_end]))
odds_aou_afr_ = pad_and_concat(odds_aou_afr_, np.diag(top_or_dict['afr'][phen][i_strt:i_end, i_strt:i_end]), np.diag(top_or_dict['afr'][phen][j_strt:j_end, j_strt:j_end]))


x_labels = ['All', '']+opt.sync_labels[0:3]
y_labels = ['All', '']+['Alc. Int.: Low', 'Alc. Int.: Med.','Alc. Int.: High',] #opt.sync_labels[i_strt-2:i_end-2]

fig, axes = plt.subplots(1, 3, figsize=(18, 10))

heatmaps = [(odds_ukb_, f'{phen}\n(UKB Eur.; Top 5%)'),
            (odds_aou_eur_, f'{phen}\n(AoU Eur.; Top 5%)'), 
            (odds_aou_afr_, f'{phen}\n(AoU Afr.; Top 5%)'), ]

plt.rcParams.update({'font.size': 20})
for idx, (ax, (data, title)) in enumerate(zip(axes, heatmaps)):
    sns.heatmap(data, ax=ax, square=True, cbar=True, annot=True, fmt=".2f", annot_kws={"size": 22},
                cmap="coolwarm", xticklabels=x_labels, 
                yticklabels=(y_labels if idx == 0 else False), cbar_kws={'shrink': 0.4,})
    
    # Add black borders around the heatmap
    ax.hlines(0, 2, data.shape[1], color="black", linewidth=8)
    ax.hlines(1, 2, data.shape[1], color="black", linewidth=4)
    ax.hlines(2, 2, data.shape[1], color="black", linewidth=4)
    ax.hlines(data.shape[0], 2, data.shape[1], color="black", linewidth=8)
    
    
    ax.vlines(0, 2, data.shape[0], color="black", linewidth=8)
    ax.vlines(1, 2, data.shape[0], color="black", linewidth=4)
    ax.vlines(2, 2, data.shape[0], color="black", linewidth=4)
    ax.vlines(data.shape[1], 2, data.shape[0], color="black", linewidth=8)
    
    ax.hlines(2, 0, 1, color="black", linewidth=4)
    ax.hlines(data.shape[0], 0, 1, color="black", linewidth=8)

    ax.vlines(2, 0, 1, color="black", linewidth=4)
    ax.vlines(data.shape[0], 0, 1, color="black", linewidth=8)

    cbar = ax.collections[0].colorbar
    if idx==2:
        cbar.ax.set_ylabel(f'Odds Ratio', fontsize=24)  # Add cbar title
    ax.set_title(title, fontsize=24)

fig.suptitle('Non-Corresponding Odds Ratios', fontsize=34, fontweight='bold', y=.775)

plt.tight_layout()
plt.savefig(f'{results_dir}/discordance.png', bbox_inches='tight')
plt.show()

In [ ]:
phen = 'CAD'
i = 4
j = 0

i_strt = bounds[i]
i_end = bounds[i+1]

j_strt = bounds[j]
j_end = bounds[j+1]

odds_ukb_ = top_or_dict['ukb'][phen][i_strt:i_end, j_strt:j_end]
odds_aou_eur_ = top_or_dict['eur'][phen][i_strt:i_end, j_strt:j_end]
odds_aou_afr_ = top_or_dict['afr'][phen][i_strt:i_end, j_strt:j_end]

odds_ukb_ = pad_and_concat(odds_ukb_,  np.diag(top_or_dict['ukb'][phen][i_strt:i_end, i_strt:i_end]), np.diag(top_or_dict['ukb'][phen][j_strt:j_end, j_strt:j_end]))
odds_aou_eur_ = pad_and_concat(odds_aou_eur_, np.diag(top_or_dict['eur'][phen][i_strt:i_end, i_strt:i_end]), np.diag(top_or_dict['eur'][phen][j_strt:j_end, j_strt:j_end]))
odds_aou_afr_ = pad_and_concat(odds_aou_afr_, np.diag(top_or_dict['afr'][phen][i_strt:i_end, i_strt:i_end]), np.diag(top_or_dict['afr'][phen][j_strt:j_end, j_strt:j_end]))


x_labels = ['All', '', 'Female', 'Male']
y_labels = ['All', '']+['Inc.: Low', 'Inc.: Med.', 'Inc.: High']#opt.sync_labels[i_strt-2:i_end-2]

fig, axes = plt.subplots(1, 3, figsize=(14, 6))

heatmaps = [(odds_ukb_, f'{phen}\n(UKB Eur., Top 5%)'), 
            (odds_aou_eur_, f'{phen}\n(AoU Eur., Top 5%)'), 
            (odds_aou_afr_, f'{phen}\n(AoU Afr., Top 5%)'), ]


for idx, (ax, (data, title)) in enumerate(zip(axes, heatmaps)):
    sns.heatmap(data, ax=ax, square=True, cbar=True, annot=True, fmt=".2f", annot_kws={"size": 18},
                cmap="coolwarm", xticklabels=x_labels, 
                yticklabels=(y_labels if idx == 0 else False), cbar_kws={'shrink': 0.8,})
    
    # Add black borders around the heatmap
    ax.hlines(0, 2, data.shape[1], color="black", linewidth=8)
    ax.hlines(1, 2, data.shape[1], color="black", linewidth=4)
    ax.hlines(2, 2, data.shape[1], color="black", linewidth=4)
    ax.hlines(data.shape[0], 2, data.shape[1], color="black", linewidth=8)
    
    
    ax.vlines(0, 2, data.shape[0], color="black", linewidth=8)
    ax.vlines(1, 2, data.shape[0], color="black", linewidth=4)
    ax.vlines(2, 2, data.shape[0], color="black", linewidth=4)
    ax.vlines(data.shape[1], 2, data.shape[0], color="black", linewidth=8)
    
    ax.hlines(2, 0, 1, color="black", linewidth=4)
    ax.hlines(data.shape[0], 0, 1, color="black", linewidth=8)

    ax.vlines(2, 0, 1, color="black", linewidth=4)
    ax.vlines(data.shape[0]-1, 0, 1, color="black", linewidth=8)

    cbar = ax.collections[0].colorbar
    if idx==2:
        cbar.ax.set_ylabel(f'Odds Ratio', fontsize=24)  # Add cbar title
    ax.set_title(title, fontsize=24)

fig.suptitle('Corresponding Odds Ratios', fontsize=34, fontweight='bold', y=.965)

plt.tight_layout()
plt.savefig(f'{results_dir}/concordance.png', bbox_inches='tight')
plt.show()
